# Notebook 01 — Single-stock LSTM smoke test

This notebook is an MVP integration smoke test for the current `ml_utils` public API. It is not a final experiment, does not include TCN or DLinear, and does not claim production benchmark performance.

If this notebook reveals a public API blocker, stop and return to the matching `ml_utils` module for a tested fix. Notebook 02 pooled workflow is deferred and is not handled here.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ml_utils").is_dir():
    for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
        if (candidate / "ml_utils").is_dir() and (candidate / "notebooks").is_dir():
            PROJECT_ROOT = candidate
            break
    else:
        raise RuntimeError(f"Could not locate project root from cwd={Path.cwd()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from ml_utils.config import DataConfig, ModelConfig, TrainConfig, WindowConfig
from ml_utils.dataset import WindowedClassificationDataset
from ml_utils.dataset import fit_scaler_on_train
from ml_utils.dataset import make_binary_labels_from_future_avg_return
from ml_utils.dataset import make_time_splits
from ml_utils.dataset import transform_split
from ml_utils.dataset import trim_labels_at_split_boundary
from ml_utils.metrics import always_predict_baseline_metrics
from ml_utils.metrics import dummy_baseline_metrics
from ml_utils.models.lstm_classifier import LSTMClassifier
from ml_utils.seed import seed_everything
from ml_utils.trainer import Trainer, evaluate

In [ ]:
SMOKE_MODE = True
MAX_ROWS_PER_TICKER = 5000
NUM_EPOCHS_SMALL = 2
SEED = 42
TICKER = "CSCO"
DATA_PATH = Path("data/CSCO.csv")
FEATURE_COLS = ["open", "high", "low", "close", "volume"]
TICKER_COL = "ticker"
LABEL_COL = "label"
SCALER_TYPE = "standard"
NUM_WORKERS = 0
CHECKPOINT_DIR = Path("checkpoints/notebook_01_lstm_smoke")

seed_everything(SEED, deterministic=True)

data_config = DataConfig(
    tickers=[TICKER],
    data_dir=str(DATA_PATH.parent),
    timestamp_col="timestamp",
    price_col="close",
    feature_cols=FEATURE_COLS,
    train_ratio=0.7,
    val_ratio=0.15,
    timezone_policy="naive",
)
window_config = WindowConfig(window_size=24, label_horizon_k=12, stride=1)
train_config = TrainConfig(
    batch_size=32,
    num_epochs=NUM_EPOCHS_SMALL,
    learning_rate=1e-3,
    weight_decay=0.0,
    grad_clip=None,
    early_stop_patience=NUM_EPOCHS_SMALL,
    monitor_metric="val_macro_f1",
    monitor_mode="max",
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=SEED,
)
model_config = ModelConfig(
    name="lstm_classifier",
    params={
        "input_size": len(FEATURE_COLS),
        "hidden_size": 32,
        "num_layers": 1,
        "num_classes": 2,
        "dropout": 0.0,
        "bidirectional": False,
    },
)

print(
    "smoke config | "
    f"seed={SEED} | ticker={TICKER} | rows_cap={MAX_ROWS_PER_TICKER} | "
    f"epochs={train_config.num_epochs} | device={train_config.device} | "
    f"features={len(FEATURE_COLS)}"
)

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"DATA_PATH does not exist: {DATA_PATH}. Set DATA_PATH to a real single-ticker CSV."
    )

raw_df = pd.read_csv(DATA_PATH)
if data_config.timestamp_col not in raw_df.columns:
    raise ValueError(f"missing timestamp column: {data_config.timestamp_col!r}")

raw_df[data_config.timestamp_col] = pd.to_datetime(
    raw_df[data_config.timestamp_col],
    errors="raise",
)
if TICKER_COL in raw_df.columns:
    observed_tickers = raw_df[TICKER_COL].dropna().unique()
    if len(observed_tickers) != 1:
        raise ValueError(
            "Notebook 01 requires a single ticker CSV; "
            f"found {len(observed_tickers)} non-null ticker values in {TICKER_COL!r}."
        )
else:
    raw_df[TICKER_COL] = TICKER

if SMOKE_MODE:
    raw_df = raw_df.iloc[:MAX_ROWS_PER_TICKER].copy(deep=True)

print(
    "loaded data | "
    f"rows={len(raw_df)} | "
    f"date_min={raw_df[data_config.timestamp_col].min()} | "
    f"date_max={raw_df[data_config.timestamp_col].max()} | "
    f"features={len(data_config.feature_cols)}"
)

In [ ]:
required_cols = [
    data_config.timestamp_col,
    data_config.price_col,
    *data_config.feature_cols,
    TICKER_COL,
]
missing_cols = [column for column in required_cols if column not in raw_df.columns]
if missing_cols:
    raise ValueError(f"missing required columns: {missing_cols}")
if not pd.api.types.is_datetime64_any_dtype(raw_df[data_config.timestamp_col]):
    raise ValueError(f"{data_config.timestamp_col!r} must be datetime dtype")
if raw_df[data_config.timestamp_col].isna().any():
    raise ValueError(f"{data_config.timestamp_col!r} contains NaN after datetime parsing")
if raw_df[data_config.timestamp_col].duplicated().any():
    raise ValueError(f"{data_config.timestamp_col!r} contains duplicate timestamps")
if not raw_df[data_config.timestamp_col].is_monotonic_increasing:
    raise ValueError(f"{data_config.timestamp_col!r} must be strictly increasing")
if (raw_df[data_config.price_col] <= 0).any():
    raise ValueError(f"{data_config.price_col!r} must be strictly positive")
if raw_df[data_config.feature_cols].isna().any().any():
    raise ValueError("feature columns contain NaN")
non_numeric_features = [
    column for column in data_config.feature_cols
    if not pd.api.types.is_numeric_dtype(raw_df[column])
]
if non_numeric_features:
    raise ValueError(f"feature columns must be numeric: {non_numeric_features}")

print(
    "schema sanity | "
    f"schema_ok=True | rows={len(raw_df)} | "
    f"feature_cols={len(data_config.feature_cols)} | ticker_col={TICKER_COL!r}"
)

In [ ]:
labeled_df = make_binary_labels_from_future_avg_return(
    raw_df,
    price_col=data_config.price_col,
    k=window_config.label_horizon_k,
)
label_counts = labeled_df[LABEL_COL].value_counts(dropna=False).sort_index()
tail_nan_count = int(labeled_df[LABEL_COL].tail(window_config.label_horizon_k).isna().sum())

print(
    "labels | "
    f"class_counts={label_counts.to_dict()} | "
    f"tail_nan_count={tail_nan_count} | "
    "class_0=non_up | class_1=up"
)

In [ ]:
train_df, val_df, test_df = make_time_splits(
    labeled_df,
    train_ratio=data_config.train_ratio,
    val_ratio=data_config.val_ratio,
    timestamp_col=data_config.timestamp_col,
    timezone_policy=data_config.timezone_policy,
)

scaler = fit_scaler_on_train(
    train_df,
    feature_cols=data_config.feature_cols,
    scaler_type=SCALER_TYPE,
)
train_scaled = transform_split(train_df, scaler, data_config.feature_cols)
val_scaled = transform_split(val_df, scaler, data_config.feature_cols)
test_scaled = transform_split(test_df, scaler, data_config.feature_cols)

train_ready = trim_labels_at_split_boundary(
    train_scaled,
    label_horizon_k=window_config.label_horizon_k,
    label_col=LABEL_COL,
    timestamp_col=data_config.timestamp_col,
    ticker_col=TICKER_COL,
    timezone_policy=data_config.timezone_policy,
)
val_ready = trim_labels_at_split_boundary(
    val_scaled,
    label_horizon_k=window_config.label_horizon_k,
    label_col=LABEL_COL,
    timestamp_col=data_config.timestamp_col,
    ticker_col=TICKER_COL,
    timezone_policy=data_config.timezone_policy,
)
test_ready = trim_labels_at_split_boundary(
    test_scaled,
    label_horizon_k=window_config.label_horizon_k,
    label_col=LABEL_COL,
    timestamp_col=data_config.timestamp_col,
    ticker_col=TICKER_COL,
    timezone_policy=data_config.timezone_policy,
)

print(
    "split/scaler/trim | "
    f"split_sizes={{'train': {len(train_ready)}, 'val': {len(val_ready)}, 'test': {len(test_ready)}}} | "
    f"valid_labels={{'train': {int(train_ready[LABEL_COL].notna().sum())}, "
    f"'val': {int(val_ready[LABEL_COL].notna().sum())}, "
    f"'test': {int(test_ready[LABEL_COL].notna().sum())}}} | "
    "scaler_fit=train_only"
)

In [ ]:
train_dataset = WindowedClassificationDataset(
    train_ready,
    feature_cols=data_config.feature_cols,
    label_col=LABEL_COL,
    ticker_col=TICKER_COL,
    timestamp_col=data_config.timestamp_col,
    window_size=window_config.window_size,
    label_horizon_k=window_config.label_horizon_k,
    stride=window_config.stride,
)
val_dataset = WindowedClassificationDataset(
    val_ready,
    feature_cols=data_config.feature_cols,
    label_col=LABEL_COL,
    ticker_col=TICKER_COL,
    timestamp_col=data_config.timestamp_col,
    window_size=window_config.window_size,
    label_horizon_k=window_config.label_horizon_k,
    stride=window_config.stride,
)
test_dataset = WindowedClassificationDataset(
    test_ready,
    feature_cols=data_config.feature_cols,
    label_col=LABEL_COL,
    ticker_col=TICKER_COL,
    timestamp_col=data_config.timestamp_col,
    window_size=window_config.window_size,
    label_horizon_k=window_config.label_horizon_k,
    stride=window_config.stride,
)

if min(len(train_dataset), len(val_dataset), len(test_dataset)) == 0:
    raise ValueError(
        "one or more datasets have zero windows; adjust DATA_PATH, MAX_ROWS_PER_TICKER, "
        "window_size, or label_horizon_k"
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=train_config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=train_config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=train_config.batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

sample_x, sample_y = next(iter(train_loader))
print(
    "datasets/loaders | "
    f"dataset_lengths={{'train': {len(train_dataset)}, 'val': {len(val_dataset)}, 'test': {len(test_dataset)}}} | "
    f"batch_x_shape={tuple(sample_x.shape)} | batch_y_shape={tuple(sample_y.shape)}"
)

In [ ]:
model = LSTMClassifier(**model_config.params)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=train_config.learning_rate,
    weight_decay=train_config.weight_decay,
)
criterion = torch.nn.CrossEntropyLoss()
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=None,
    device=train_config.device,
    checkpoint_dir=str(CHECKPOINT_DIR),
    monitor_metric=train_config.monitor_metric,
    monitor_mode=train_config.monitor_mode,
    early_stop_patience=train_config.early_stop_patience,
    grad_clip=train_config.grad_clip,
    verbose=True,
)

param_count = sum(parameter.numel() for parameter in model.parameters())
print(
    "model/trainer | "
    f"model={model_config.name} | parameters={param_count} | "
    f"device={train_config.device} | checkpoint_dir={CHECKPOINT_DIR}"
)

In [ ]:
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=train_config.num_epochs,
)

print(
    "training complete | "
    f"best_epoch={history['best_epoch']} | "
    f"best_{train_config.monitor_metric}={history['best_metric']:.6f}"
)

In [ ]:
val_metrics, val_true, val_pred = evaluate(
    model,
    val_loader,
    criterion,
    train_config.device,
)
test_metrics, test_true, test_pred = evaluate(
    model,
    test_loader,
    criterion,
    train_config.device,
)

confusion_matrix_df = pd.DataFrame(
    test_metrics["confusion_matrix"],
    index=["true_non_up", "true_up"],
    columns=["pred_non_up", "pred_up"],
)
print(
    "evaluation | "
    f"val_macro_f1={val_metrics['macro_f1']:.6f} | "
    f"test_macro_f1={test_metrics['macro_f1']:.6f} | "
    f"test_balanced_accuracy={test_metrics['balanced_accuracy']:.6f}"
)
confusion_matrix_df

In [ ]:
y_train = np.array([int(train_dataset[index][1].item()) for index in range(len(train_dataset))])
eval_labels_by_split = {
    "val": val_true,
    "test": test_true,
}

baseline_rows = []
for split_name, y_eval in eval_labels_by_split.items():
    stratified_runs = [
        dummy_baseline_metrics(
            y_train=y_train,
            y_test=y_eval,
            strategy="stratified",
            random_state=SEED + run_index,
        )
        for run_index in range(10)
    ]
    baseline_rows.append(
        {
            "model_name": "dummy_stratified",
            "split": split_name,
            "accuracy": float(np.mean([run["accuracy"] for run in stratified_runs])),
            "macro_f1": float(np.mean([run["macro_f1"] for run in stratified_runs])),
            "balanced_accuracy": float(np.mean([run["balanced_accuracy"] for run in stratified_runs])),
            "macro_f1_std": float(np.std([run["macro_f1"] for run in stratified_runs], ddof=0)),
            "notes": "mean over 10 seeds",
        }
    )

    prior_metrics = dummy_baseline_metrics(
        y_train=y_train,
        y_test=y_eval,
        strategy="prior",
        random_state=SEED,
    )
    baseline_rows.append(
        {
            "model_name": "dummy_prior",
            "split": split_name,
            "accuracy": prior_metrics["accuracy"],
            "macro_f1": prior_metrics["macro_f1"],
            "balanced_accuracy": prior_metrics["balanced_accuracy"],
            "macro_f1_std": np.nan,
            "notes": "fit on train labels",
        }
    )

    for model_name, constant_label in (("always_up", 1), ("always_down", 0)):
        constant_metrics = always_predict_baseline_metrics(
            y_test=y_eval,
            constant_label=constant_label,
        )
        baseline_rows.append(
            {
                "model_name": model_name,
                "split": split_name,
                "accuracy": constant_metrics["accuracy"],
                "macro_f1": constant_metrics["macro_f1"],
                "balanced_accuracy": constant_metrics["balanced_accuracy"],
                "macro_f1_std": np.nan,
                "notes": f"constant_label={constant_label}",
            }
        )

baseline_table = pd.DataFrame(baseline_rows)
baseline_table

In [ ]:
model_rows = pd.DataFrame(
    [
        {
            "model_name": "lstm_classifier",
            "split": "val",
            "accuracy": val_metrics["accuracy"],
            "macro_f1": val_metrics["macro_f1"],
            "balanced_accuracy": val_metrics["balanced_accuracy"],
            "macro_f1_std": np.nan,
            "notes": "single-stock smoke",
        },
        {
            "model_name": "lstm_classifier",
            "split": "test",
            "accuracy": test_metrics["accuracy"],
            "macro_f1": test_metrics["macro_f1"],
            "balanced_accuracy": test_metrics["balanced_accuracy"],
            "macro_f1_std": np.nan,
            "notes": "single-stock smoke",
        },
    ]
)
comparison_table = pd.concat([model_rows, baseline_table], ignore_index=True)
comparison_table["delta_macro_f1_vs_dummy_stratified"] = np.nan
for split_name in comparison_table["split"].unique():
    dummy_macro_f1 = comparison_table.loc[
        (comparison_table["split"] == split_name)
        & (comparison_table["model_name"] == "dummy_stratified"),
        "macro_f1",
    ].iloc[0]
    split_mask = comparison_table["split"] == split_name
    comparison_table.loc[split_mask, "delta_macro_f1_vs_dummy_stratified"] = (
        comparison_table.loc[split_mask, "macro_f1"] - dummy_macro_f1
    )

comparison_table = comparison_table[
    [
        "model_name",
        "split",
        "accuracy",
        "macro_f1",
        "balanced_accuracy",
        "delta_macro_f1_vs_dummy_stratified",
        "notes",
    ]
]
comparison_table

## Known limitations

- Single-stock only.
- Smoke mode uses a small row cap and a small epoch count by default.
- This is not a final benchmark and is not a claim of predictive performance.
- TCN and DLinear are not included here.
- API blockers must be fixed in `ml_utils`, not worked around in the notebook.
- Notebook 02 pooled workflow is deferred.
- Pooled global scaler API support is not validated in Notebook 01.
- Shuffled-label sanity check belongs to Notebook 02, not Notebook 01.
- Notebook 02 shuffled-label sanity threshold will use the dummy-stratified baseline ratio, not a vague random-level claim.